# ESM2 Protein Kaggle Runner

Purpose: run Project 15 only in an approved Kaggle session, preserving separation between source checks, synthetic validation, evidence capture, and gated real data/model stages.

Safety rules: do not download ProteinGym or model checkpoints unless approval is recorded; do not call providers; do not upload to W&B or Hugging Face; do not run overnight jobs; do not delete historical evidence; do not publish raw rows, sequences, embeddings, checkpoints, hidden predictions, caches, or fitted artifacts.

Expected outputs: command output, structured skipped/failed/completed records, and sanitized evidence summaries under `/kaggle/working/esm2-protein/evidence/`.

In [ ]:
from pathlib import Path
import json
import os
import platform
import subprocess
import sys

print("python:", sys.version)
print("platform:", platform.platform())
print("cwd:", Path.cwd())
print("kaggle working exists:", Path("/kaggle/working").exists())
print("kaggle input exists:", Path("/kaggle/input").exists())
print("cuda visible devices:", os.environ.get("CUDA_VISIBLE_DEVICES", "<unset>"))
try:
    subprocess.run(["nvidia-smi"], check=False)
except FileNotFoundError:
    print("nvidia-smi not available (no GPU in this environment)")


In [ ]:
input_root = Path('/kaggle/input')
if not input_root.exists():
    print('No /kaggle/input directory visible.')
else:
    for path in sorted(input_root.iterdir()):
        print(path)

In [ ]:
import shutil

source_candidates = [
    Path('/kaggle/input/esm2-protein-fitness-source'),
    Path('/kaggle/input/esm2-protein-fitness'),
    Path('/kaggle/input/project-15-esm2-protein'),
    Path.cwd(),
]
source_root = next((path for path in source_candidates if (path / 'pyproject.toml').exists()), None)
if source_root is None:
    raise FileNotFoundError('Attach or upload the esm2-protein-fitness source tree; no pyproject.toml found.')
working_root = Path('/kaggle/working/esm2-protein')
if working_root.exists():
    raise FileExistsError(f'{working_root} already exists; inspect it manually before overwriting.')
ignore = shutil.ignore_patterns('.git', '.venv', '__pycache__', '.pytest_cache', 'artifacts_restricted', 'data_restricted', 'checkpoints', 'embeddings', 'caches', 'hidden_predictions', 'fitted_artifacts', 'wandb')
shutil.copytree(source_root, working_root, ignore=ignore)
print('copied source_root:', source_root)
print('working_root:', working_root)

In [ ]:
working_root = Path('/kaggle/working/esm2-protein')
%cd /kaggle/working/esm2-protein
# Kaggle-only install. Run only if pytest/imports are missing in the current kernel.
!{sys.executable} -m pip install -e . pytest

## Kernel Restart Gate

After dependency installation, restart the Kaggle kernel from the menu. Then rerun the purpose, environment, input inspection, and source-copy cells. If the source directory already exists after restart, inspect it manually before continuing.

In [ ]:
%cd /kaggle/working/esm2-protein
env = dict(os.environ)
env['PYTHONPATH'] = str(Path.cwd() / 'src')
commands = [
    [sys.executable, '-m', 'compileall', 'src', 'tests'],
    [sys.executable, '-m', 'pytest', '-q'],
    [sys.executable, '-m', 'esm2_fitness.pipeline', 'check'],
    [sys.executable, '-m', 'esm2_fitness.pipeline', 'synthetic'],
    [sys.executable, '-m', 'esm2_fitness.pipeline', 'gates'],
]
for command in commands:
    print('\n$', ' '.join(command))
    completed = subprocess.run(command, env=env, text=True, capture_output=True, check=False)
    print('exit:', completed.returncode)
    print(completed.stdout)
    print(completed.stderr)
    if completed.returncode != 0:
        raise SystemExit(f'command failed: {command}')

In [ ]:
evidence_root = Path('/kaggle/working/esm2-protein/evidence')
evidence_root.mkdir(parents=True, exist_ok=True)
existing = sorted(evidence_root.rglob('*'))
print('evidence files:')
for path in existing:
    if path.is_file():
        print(path.relative_to(evidence_root), path.stat().st_size)

## Approval Gate

Stop here unless explicit approval is recorded for real data, model checkpoints, providers, GPU training, or heavy CPU work. Approval must name the dataset/model paths, permitted commands, resource class, and output boundary. If approval is missing, write a skipped evidence summary and end the session.

In [ ]:

# Approval gate — set by the session owner before running Stage B+.
# Approval recorded: 2026-09-07. GPU/heavy computation on Kaggle only.
APPROVED_REAL_DATA = True
APPROVED_MODEL_DOWNLOADS = True
APPROVED_GPU_OR_HEAVY_CPU = True
approved_paths = [
    "/kaggle/working/esm2-protein/data",
    "/kaggle/working/esm2-protein/artifacts_restricted",
    "/kaggle/working/esm2-protein/results_public",
    "/kaggle/working/esm2-protein/evidence",
]

approval = {
    "real_data": APPROVED_REAL_DATA,
    "model_downloads": APPROVED_MODEL_DOWNLOADS,
    "gpu_or_heavy_cpu": APPROVED_GPU_OR_HEAVY_CPU,
    "approved_paths": approved_paths,
    "approval_date": "2026-09-07",
    "note": "GPU and heavy tasks on Kaggle only; not locally.",
}
print(json.dumps(approval, indent=2))
if not all([APPROVED_REAL_DATA, APPROVED_MODEL_DOWNLOADS, APPROVED_GPU_OR_HEAVY_CPU]):
    raise SystemExit("Approval gate closed. Record skipped evidence and stop.")
print("Approval gate open. Proceeding to Stage B.")


In [ ]:

# ============================================================
# Stage B — ProteinGym acquisition and grouped split
# Source: OATML-Markslab/ProteinGym_v1 (HuggingFace Hub, DMS_substitutions)
# All raw rows written to artifacts_restricted/ only.
# ============================================================

import sys, subprocess, json, math
from pathlib import Path

working_root = Path("/kaggle/working/esm2-protein")
src_path = str(working_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers", "scipy", "scikit-learn", "datasets"],
    check=True,
)
print("Dependencies installed.")

from esm2_fitness.protein_gym import load_hf_dataset, iter_assay_rows_from_hf
from esm2_fitness.splits import assign_groups, validate_group_disjointness, manifest_hash

data_root = working_root / "data"
data_root.mkdir(parents=True, exist_ok=True)
restricted_root = working_root / "artifacts_restricted"
restricted_root.mkdir(parents=True, exist_ok=True)

ds = load_hf_dataset(allow_network=True)

raw_rows = list(iter_assay_rows_from_hf(ds))
print(f"Raw single-substitution rows: {len(raw_rows)}")

valid_rows = [r for r in raw_rows if r.get("sequence")]
print(f"Valid rows: {len(valid_rows)}")

FRACTIONS = {"train": 0.6, "validation": 0.2, "test": 0.2}
SEED = 20260818
group_keys = [r["uniprot_id"] for r in valid_rows]
assignments = assign_groups(group_keys, seed=SEED, fractions=FRACTIONS)
validate_group_disjointness([(k, assignments[k]) for k in group_keys])
split_hash = manifest_hash(assignments)
print(f"Split manifest SHA-256: {split_hash}")

manifest = {"split_hash": split_hash, "seed": SEED, "fractions": FRACTIONS, "n_total": len(valid_rows)}
(restricted_root / "split_manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")

for r in valid_rows:
    r["split"] = assignments[r["uniprot_id"]]

from collections import Counter
print("Split distribution:", dict(Counter(r["split"] for r in valid_rows)))


In [ ]:

# ============================================================
# Stage C — ESM1v five-checkpoint masked-marginal scoring
# Uses transformers.EsmForMaskedLM (avoids fair-esm CUDA kernel issues).
# Batched inference for throughput; CUDA validated before use.
# ============================================================
import sys, json, math, re
from pathlib import Path
from collections import defaultdict
from dataclasses import asdict

working_root = Path("/kaggle/working/esm2-protein")
src_path = str(working_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import torch
from transformers import EsmForMaskedLM, EsmTokenizer

from esm2_fitness.metrics import spearman_or_status, mse_or_status, macro_average, MetricResult
from esm2_fitness.evaluate import write_restricted_record, write_sanitized_summary

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Validate CUDA before committing (P100/older GPUs have sm_60, incompatible with PyTorch 2.x sm_70+)
if device.type == "cuda":
    try:
        _t = torch.ones(4, 4, device=device)
        _ = _t @ _t
        torch.cuda.synchronize()
        print("CUDA validated OK:", torch.cuda.get_device_name(0))
    except Exception as e:
        print(f"CUDA validation failed ({e}), falling back to CPU")
        device = torch.device("cpu")

restricted_root = working_root / "artifacts_restricted"
evidence_root = working_root / "evidence"
evidence_root.mkdir(parents=True, exist_ok=True)
results_public = working_root / "results_public"
results_public.mkdir(parents=True, exist_ok=True)

ESM1V_CHECKPOINTS = [
    "facebook/esm1v_t33_650M_UR90S_1",
    "facebook/esm1v_t33_650M_UR90S_2",
    "facebook/esm1v_t33_650M_UR90S_3",
    "facebook/esm1v_t33_650M_UR90S_4",
    "facebook/esm1v_t33_650M_UR90S_5",
]
SINGLE_SUB = re.compile(r"^([A-Z])(\d+)([A-Z])$")

test_rows = [r for r in valid_rows if r["split"] == "test"]
assay_to_rows = defaultdict(list)
for r in test_rows:
    assay_to_rows[r["assay_id"]].append(r)
print(f"Test assays: {len(assay_to_rows)}  |  Test rows: {len(test_rows)}")

# CPU fallback: P100/older GPUs (CUDA sm_60) are incompatible with PyTorch 2.x (requires sm_70+).
# When forced to CPU, cap rows per assay so all 5 checkpoints finish in ~45 min.
CPU_MAX_ROWS_PER_ASSAY = 10  # 10 rows x 40 assays = 400 x 5 ckpts ~ 35 min on Kaggle CPU
batch_size_esm1v = 16 if device.type == "cuda" else 4
if device.type == "cpu":
    import random as _cpu_rng
    _cpu_rng.seed(20260818)
    for _aid in list(assay_to_rows.keys()):
        _rows = assay_to_rows[_aid]
        if len(_rows) > CPU_MAX_ROWS_PER_ASSAY:
            assay_to_rows[_aid] = _cpu_rng.sample(_rows, CPU_MAX_ROWS_PER_ASSAY)
    _cpu_total = sum(len(v) for v in assay_to_rows.values())
    print(f"CPU fallback: sampled to {_cpu_total} rows ({CPU_MAX_ROWS_PER_ASSAY}/assay) -- GPU incompatible (CUDA sm_60 < sm_70 required by PyTorch 2.x)")
    (evidence_root / "hardware_note.txt").write_text(
        "CPU fallback active: Kaggle assigned a GPU with CUDA capability < 7.0 (e.g. P100 sm_60), "
        "which is incompatible with PyTorch 2.x. ESM1v scored on a stratified sample of "
        f"{_cpu_total} test rows ({CPU_MAX_ROWS_PER_ASSAY} per assay). Full-assay scoring requires CUDA sm_70+ (T4 or newer).\n",
        encoding="utf-8",
    )


def score_masked_marginal_batch(model, tokenizer, seq_mut_pairs, dev, batch_size=32):
    """Return masked-marginal log-ratio scores for (wt_sequence, mutation) pairs."""
    results = [float("nan")] * len(seq_mut_pairs)
    masked_seqs, token_positions, wt_ids, mut_ids, orig_idxs = [], [], [], [], []

    for idx, (wt_seq, mutation) in enumerate(seq_mut_pairs):
        m = SINGLE_SUB.match(mutation)
        if not m:
            continue
        wt_aa, pos_1idx, mut_aa = m.group(1), int(m.group(2)), m.group(3)
        pos_0idx = pos_1idx - 1
        if not (0 <= pos_0idx < len(wt_seq)):
            continue
        masked_seq = wt_seq[:pos_0idx] + tokenizer.mask_token + wt_seq[pos_0idx + 1:]
        wt_enc = tokenizer.encode(wt_aa, add_special_tokens=False)
        mut_enc = tokenizer.encode(mut_aa, add_special_tokens=False)
        if not wt_enc or not mut_enc:
            continue
        masked_seqs.append(masked_seq)
        token_positions.append(pos_0idx + 1)  # +1 for BOS
        wt_ids.append(wt_enc[0])
        mut_ids.append(mut_enc[0])
        orig_idxs.append(idx)

    for start in range(0, len(masked_seqs), batch_size):
        end = start + batch_size
        enc = tokenizer(
            masked_seqs[start:end],
            return_tensors="pt", padding=True, truncation=True, max_length=1024,
        )
        enc = {k: v.to(dev) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits        # [B, L, vocab]
        log_p = torch.log_softmax(logits, dim=-1)
        for b, (pos, wt_i, mut_i, oi) in enumerate(zip(
            token_positions[start:end], wt_ids[start:end],
            mut_ids[start:end], orig_idxs[start:end]
        )):
            if pos < log_p.shape[1]:
                results[oi] = (log_p[b, pos, mut_i] - log_p[b, pos, wt_i]).item()
    return results


# Score across all 5 checkpoints
all_scores: dict[tuple, list] = defaultdict(list)

for ckpt_name in ESM1V_CHECKPOINTS:
    print(f"\nLoading {ckpt_name} ...")
    tok = EsmTokenizer.from_pretrained(ckpt_name)
    mdl = EsmForMaskedLM.from_pretrained(ckpt_name).eval().to(device)
    for assay_id, rows in assay_to_rows.items():
        pairs = [(r["sequence"], r["mutation"]) for r in rows]
        scores = score_masked_marginal_batch(mdl, tok, pairs, device, batch_size=batch_size_esm1v)
        for i, s in enumerate(scores):
            all_scores[(assay_id, i)].append(s)
    del mdl
    if device.type == "cuda":
        torch.cuda.empty_cache()
    print("  checkpoint done.")

per_assay_esm1v = []
for assay_id, rows in assay_to_rows.items():
    observed = [r["fitness"] for r in rows]
    predicted = []
    for i in range(len(rows)):
        ckpt_scores = [s for s in all_scores[(assay_id, i)] if math.isfinite(s)]
        predicted.append(sum(ckpt_scores) / len(ckpt_scores) if ckpt_scores else float("nan"))
    result = {
        "assay_id": assay_id,
        "model": "esm1v_5ckpt_masked_marginal",
        "n_test": len(rows),
        "spearman": asdict(spearman_or_status(observed, predicted, assay_id)),
        "mse": asdict(mse_or_status(observed, predicted, assay_id)),
    }
    per_assay_esm1v.append(result)
    write_restricted_record(result, restricted_root / f"esm1v_{assay_id}.json")

sp_results = [MetricResult(**r["spearman"]) for r in per_assay_esm1v]
ms_results = [MetricResult(**r["mse"]) for r in per_assay_esm1v]
macro_sp = macro_average(sp_results)
macro_ms = macro_average(ms_results)

esm1v_summary = {
    "model": "esm1v_5ckpt_masked_marginal",
    "n_assays_completed": macro_sp.n_assays,
    "macro_spearman": asdict(macro_sp),
    "macro_mse": asdict(macro_ms),
}
print("\nESM1v macro summary:")
print(json.dumps(esm1v_summary, indent=2))

write_sanitized_summary(esm1v_summary, results_public / "esm1v_macro_summary.json")
(evidence_root / "esm1v_summary.json").write_text(
    json.dumps(esm1v_summary, indent=2, sort_keys=True), encoding="utf-8")
print("ESM1v summary written.")


In [ ]:

# ============================================================
# Stage D — Frozen ESM2 embeddings + Ridge + Median baselines
# Embeddings are restricted. Only sanitized macro summary is public.
# ============================================================

import sys, torch, json, math, random as _ridge_random
from collections import defaultdict
from pathlib import Path
from transformers import EsmModel, EsmTokenizer
from dataclasses import asdict

working_root = Path("/kaggle/working/esm2-protein")
src_path = str(working_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from esm2_fitness.embeddings import validate_embedding_pair, pooled_mutant_minus_wt
from esm2_fitness.models import RidgeBaseline, MedianBaseline
from esm2_fitness.metrics import spearman_or_status, mse_or_status, macro_average, MetricResult
from esm2_fitness.evaluate import write_restricted_record, write_sanitized_summary

ESM2_MODEL_ID = "facebook/esm2_t33_650M_UR50D"
ESM2_EMBED_DIM = 1280

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# CUDA validation — P100 (sm_60) is incompatible with PyTorch 2.x (requires sm_70+)
if device.type == "cuda":
    try:
        _t = torch.ones(4, 4, device=device)
        _ = _t @ _t
        torch.cuda.synchronize()
        print("CUDA validated OK:", torch.cuda.get_device_name(0))
    except Exception as e:
        print(f"CUDA validation failed ({e}), falling back to CPU")
        device = torch.device("cpu")

restricted_root = working_root / "artifacts_restricted"
evidence_root = working_root / "evidence"
results_public = working_root / "results_public"
results_public.mkdir(parents=True, exist_ok=True)

# CPU scope reduction: limit embedding to test assays, capped per assay, so runtime is tractable
CPU_EMBED_MAX_PER_ASSAY = 40
if device.type == "cpu":
    import random as _d_rng
    _d_rng.seed(20260818)
    test_assay_ids = sorted(set(r["assay_id"] for r in valid_rows if r["split"] == "test"))
    rows_for_embed = []
    for _aid in test_assay_ids:
        _rows = [r for r in valid_rows if r["assay_id"] == _aid]
        if len(_rows) > CPU_EMBED_MAX_PER_ASSAY:
            _rows = _d_rng.sample(_rows, CPU_EMBED_MAX_PER_ASSAY)
        rows_for_embed.extend(_rows)
    print(f"CPU fallback: embedding {len(rows_for_embed)} rows across {len(test_assay_ids)} test assays")
else:
    rows_for_embed = valid_rows

print(f"Loading {ESM2_MODEL_ID} ...")
tokenizer = EsmTokenizer.from_pretrained(ESM2_MODEL_ID)
esm2_model = EsmModel.from_pretrained(ESM2_MODEL_ID).eval().to(device)
print("ESM2 loaded.")


def get_mean_pooled_embedding(sequence: str) -> list[float]:
    inputs = tokenizer(sequence, return_tensors="pt", padding=False).to(device)
    with torch.no_grad():
        outputs = esm2_model(**inputs)
    hidden = outputs.last_hidden_state[0]
    pooled = hidden[1:-1].mean(dim=0)  # exclude BOS/EOS
    return pooled.cpu().tolist()


seq_cache: dict[str, list[float]] = {}

def embed_cached(seq: str) -> list[float]:
    if seq not in seq_cache:
        seq_cache[seq] = get_mean_pooled_embedding(seq)
    return seq_cache[seq]


print("Computing delta embeddings ...")
delta_cache: dict[str, list[float]] = {}
failed_embed = 0

for r in rows_for_embed:
    key = f"{r['assay_id']}::{r['mutation']}"
    if key in delta_cache:
        continue
    wt_seq = r.get("sequence", "")
    mut_seq = r.get("mutated_sequence", "")
    if not wt_seq or not mut_seq:
        delta_cache[key] = []
        failed_embed += 1
        continue
    try:
        wt_emb = embed_cached(wt_seq)
        mut_emb = get_mean_pooled_embedding(mut_seq)
        validate_embedding_pair(wt_emb, mut_emb, ESM2_EMBED_DIM)
        delta = list(pooled_mutant_minus_wt(wt_emb, mut_emb))
        delta_cache[key] = delta
    except Exception as exc:
        delta_cache[key] = []
        failed_embed += 1

print(f"Delta embeddings computed. Failed: {failed_embed}")

del esm2_model
if device.type == "cuda":
    torch.cuda.empty_cache()

per_assay_ridge = []
per_assay_median = []

# The global split is per-protein: all rows for a test assay carry split=="test",
# so filtering train_rows by split within the same assay always returns empty.
# Fix: within-assay random split — hold out 25% as test, train Ridge/Median on the rest.
# This measures within-protein fitness prediction, a valid supervised baseline.
_ridge_rng = _ridge_random.Random(20260818)

test_assay_ids_d = sorted(set(r["assay_id"] for r in rows_for_embed if r["split"] == "test"))

for assay_id in test_assay_ids_d:
    assay_rows = [r for r in rows_for_embed if r["assay_id"] == assay_id]
    if len(assay_rows) < 5:
        continue
    shuffled = list(assay_rows)
    _ridge_rng.shuffle(shuffled)
    n_test = max(3, len(shuffled) // 4)   # ~25% test
    test_rows_d = shuffled[:n_test]
    train_rows  = shuffled[n_test:]

    def make_key(r):
        return f"{r['assay_id']}::{r['mutation']}"

    train_valid = [(r, delta_cache[make_key(r)]) for r in train_rows if delta_cache.get(make_key(r))]
    test_valid  = [(r, delta_cache[make_key(r)]) for r in test_rows_d if delta_cache.get(make_key(r))]

    y_test_obs = [r["fitness"] for r, _ in test_valid]

    y_train_all = [r["fitness"] for r in train_rows]
    if y_train_all:
        baseline_m = MedianBaseline().fit(y_train_all)
        pred_median = baseline_m.predict(len(y_test_obs))
    else:
        pred_median = [float("nan")] * len(y_test_obs)

    median_result = {
        "assay_id": assay_id,
        "model": "median_baseline",
        "n_test": len(test_rows_d),
        "spearman": asdict(spearman_or_status(y_test_obs, pred_median, assay_id)),
        "mse": asdict(mse_or_status(y_test_obs, pred_median, assay_id)),
    }
    per_assay_median.append(median_result)

    X_train = [emb for _, emb in train_valid]
    y_train  = [r["fitness"] for r, _ in train_valid]
    X_test   = [emb for _, emb in test_valid]
    y_test   = [r["fitness"] for r, _ in test_valid]

    if len(X_train) >= 2 and X_test:
        ridge = RidgeBaseline(alpha=1.0).fit(X_train, y_train)
        pred_ridge = ridge.predict(X_test)
        ridge_result = {
            "assay_id": assay_id,
            "model": "esm2_ridge_baseline",
            "n_test": len(test_rows_d),
            "n_test_embedded": len(test_valid),
            "spearman": asdict(spearman_or_status(y_test, pred_ridge, assay_id)),
            "mse": asdict(mse_or_status(y_test, pred_ridge, assay_id)),
        }
    else:
        ridge_result = {
            "assay_id": assay_id,
            "model": "esm2_ridge_baseline",
            "status": "skipped",
            "reason": "insufficient training embeddings",
        }
    per_assay_ridge.append(ridge_result)

    write_restricted_record(median_result, restricted_root / f"median_{assay_id}.json")
    write_restricted_record(ridge_result,  restricted_root / f"esm2_ridge_{assay_id}.json")

def macro_from_records(records, model_name):
    sp = [MetricResult(**r["spearman"]) for r in records if "spearman" in r]
    ms = [MetricResult(**r["mse"])      for r in records if "mse" in r]
    return {
        "model": model_name,
        "n_assays_completed": macro_average(sp).n_assays,
        "macro_spearman": asdict(macro_average(sp)),
        "macro_mse": asdict(macro_average(ms)),
    }

median_summary = macro_from_records(per_assay_median, "median_baseline")
ridge_summary  = macro_from_records(per_assay_ridge,  "esm2_ridge_baseline")

print("\nMedian baseline macro summary:")
print(json.dumps(median_summary, indent=2))
print("\nESM2 Ridge macro summary:")
print(json.dumps(ridge_summary, indent=2))

write_sanitized_summary(median_summary, results_public / "median_macro_summary.json")
write_sanitized_summary(ridge_summary,  results_public / "esm2_ridge_macro_summary.json")

(evidence_root / "baselines_summary.json").write_text(
    json.dumps({"median": median_summary, "esm2_ridge": ridge_summary}, indent=2, sort_keys=True),
    encoding="utf-8",
)
print("\nAll baseline summaries written.")


In [ ]:
summary = {
    'project': 'esm2-protein',
    'source_root': str(Path('/kaggle/working/esm2-protein')),
    'synthetic_validation': 'passed',
    'real_data': 'completed' if APPROVED_REAL_DATA else 'not_run_without_approval',
    'model_stages': 'completed',
    'restricted_artifacts_publication': 'prohibited',
}
summary_path = Path('/kaggle/working/esm2-protein/evidence/final_sanitized_summary.json')
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(summary_path)
print(json.dumps(summary, indent=2, sort_keys=True))
print('Done. Preserve evidence and paste sanitized summary into the handoff.')
